# Day 5 — LLMOps A/B Test Dashboard
**Goal:** Streamlit dashboard routing 50/50 traffic between base Mistral-7B and NyayaGPT.
All requests logged to MLflow under `nyayagpt-ab-test`.

**Architecture:**
```
Browser → Streamlit UI → ab_generate() → [base model | fine-tuned]
                              ↓
                       MLflow (logs latency + variant per request)
```

In [ ]:
import sys, os, subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)

# Install streamlit if needed
try:
    import streamlit
    print(f'✓ streamlit {streamlit.__version__}')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'])
    print('streamlit installed')

In [ ]:
# Check prerequisites
from nyaya_pipeline import config
assert config.ADAPTER_DIR.exists(), f'Adapter not found. Run Day 2 first.'
print(f'✓ Adapter: {config.ADAPTER_DIR}')

dashboard_path = PROJECT_ROOT / 'src' / 'nyaya_pipeline' / 'dashboard.py'
assert dashboard_path.exists(), f'dashboard.py not found at {dashboard_path}'
print(f'✓ Dashboard: {dashboard_path}')

## Preview the A/B routing logic
(Without launching Streamlit)

In [ ]:
# Show the A/B routing function
import inspect
from nyaya_pipeline.infer import ab_generate
print(inspect.getsource(ab_generate))

In [ ]:
# Test A/B routing manually (without Streamlit)
# Note: loads both models — needs ~20-25 GB VRAM
question = 'What is the maximum punishment under Section 302 IPC?'

print('Testing A/B routing …')
variant, base_resp, ft_resp, base_ms, ft_ms = ab_generate(
    question=question,
    max_new_tokens=128,
)

print(f'Assigned variant: {variant}')
print(f'\nBase Mistral ({base_ms:.0f} ms):')
print(f'  {base_resp[:200]}')
print(f'\nNyayaGPT ({ft_ms:.0f} ms):')
print(f'  {ft_resp[:200]}')

## Launch Streamlit Dashboard

In [ ]:
# Also start MLflow UI if not already running
import subprocess, sys
from nyaya_pipeline import config

mlflow_proc = subprocess.Popen(
    [sys.executable, '-m', 'mlflow', 'ui',
     '--backend-store-uri', config.MLFLOW_URI,
     '--port', '5000'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print('MLflow UI started → http://localhost:5000')

In [ ]:
# Launch Streamlit dashboard
# This will block the notebook cell — open in a terminal instead for a non-blocking run:
#   cd /home/ubuntu/Fine-tuning/NyayaGPT
#   streamlit run src/nyaya_pipeline/dashboard.py --server.port 8501

print('To launch the dashboard, run this in a terminal:')
print()
print(f'  cd {PROJECT_ROOT}')
print(f'  streamlit run src/nyaya_pipeline/dashboard.py --server.port 8501')
print()
print('Then open: http://localhost:8501')

# Uncomment to launch from notebook (blocks until stopped):
# subprocess.run([sys.executable, '-m', 'streamlit', 'run',
#                 str(dashboard_path), '--server.port', '8501'])

## Dashboard UI Preview
```
┌──────────────────────────────────────────────────────────────┐
│  ⚖️  NyayaGPT — A/B Test Dashboard                           │
│  ─────────────────────────────────────────                   │
│  Question: [_______________________________] [Send ⚡]       │
│                                                              │
│  🔵 Base Mistral-7B        🟢 NyayaGPT Fine-tuned            │
│  ┌────────────────────┐   ┌────────────────────────┐         │
│  │ Response A...      │   │ Response B...           │         │
│  │ Latency: 48 ms     │   │ Latency: 42 ms          │         │
│  └────────────────────┘   └────────────────────────┘         │
│                                                              │
│  Total: 7 | NyayaGPT served: 4/7 | Avg delta: -6 ms         │
└──────────────────────────────────────────────────────────────┘
```

## ✅ Day 5 Complete

**MLflow A/B logs:** `mlruns/` → experiment `nyayagpt-ab-test`

**Next:** Run `06_hf_hub_deployment.ipynb`